## byLLM Script

Description here

In [11]:
import os
import pandas as pd
from byllm.lib import Model, by

# This pulls the value from the system variable
apiKey = os.getenv('LLM_API_KEY')

if not apiKey:
    raise ValueError("Missing API Key! Please set the LLM_API_KEY environment variable.")

llm = Model(model_name="gemini/gemini-2.5-flash" , api_key= apiKey)

## Import Datasets

In [12]:
filePath = '../../data/fdic/fdic_250.csv'
df = pd.read_csv(filePath)

df.head()

,fdic_certificate_number,institution_name,state_name,fdic_id,docket,active,address,total_assets,bank_charter_class,change_code_1,...,csa_name,csa_fips_code,csa_indicator,cbsa_name,cbsa_fips_code,cbsa_metro_flag,cbsa_micro_flag,cbsa_division_name,cbsa_division_fips_code,cbsa_division_flag
0,8870,BankFirst Financial Services,Mississippi,5663,10529,True,900 Main St,2045233.0,SM,810.0,...,"Columbus-West Point, MS",200.0,True,"Columbus, MS",18060.0,False,True,NaN,NaN,False
1,11423,Sycamore Bank,Mississippi,7294,0,False,301 E Main St,329170.0,NM,223.0,...,"Memphis-Forrest City, TN-MS-AR",368.0,True,"Memphis, TN-MS-AR",32820.0,True,False,NaN,NaN,False
2,9120,Rolette State Bank,North Dakota,5838,0,True,209 Main Street,51045.0,NM,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3914,Ramsey National Bank,North Dakota,2551,12753,False,300 4th St Ne,319992.0,N,223.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,28679,North Shore Bank,Wisconsin,41513,2130,True,15700 W Bluemound Rd,2617053.0,SI,510.0,...,"Milwaukee-Racine-Waukesha, WI",376.0,True,"Milwaukee-Waukesha, WI",33340.0,True,False,NaN,NaN,False


In [13]:
# Create df2 for a smaller sample in testing
df2 = df.iloc[:25, :5]
#df2.shape

## byLLM model integration

In [14]:
import pandas as pd
import time
from dataclasses import dataclass, asdict

@dataclass
class TextBatch:
    values: list[str]

@by(llm)
def clean_column_values(batch: TextBatch) -> TextBatch:
    """
    Clean this list of strings
    """
    ...

def clean_entire_dataframe(df: pd.DataFrame):
    cleaned_df = df.copy().astype(str)
    
    for col in df.columns:
        print(f"--- Cleaning Column: {col} ---")
        column_data = cleaned_df[col].tolist()
        new_column_values = []
        
        # Process the column in chunks of 50 values (much faster!)
        batch_size = 50 
        for i in range(0, len(column_data), batch_size):
            chunk = column_data[i : i + batch_size]
            
            try:
                # Wrap in dataclass for byllm
                result = clean_column_values(TextBatch(values=chunk))
                new_column_values.extend(result.values)
                
                # Small sleep to respect the 15 RPM limit
                time.sleep(4) 
                
            except Exception as e:
                print(f"Error in {col} at index {i}: {e}")
                new_column_values.extend(chunk) # Keep original on failure
                time.sleep(20)
        
        # Update the column in our dataframe
        # Ensure the lengths match before assignment
        if len(new_column_values) == len(cleaned_df):
            cleaned_df[col] = new_column_values
            
    return cleaned_df

# --- Execution ---
# df_fully_cleaned = clean_entire_dataframe(df2)

## Slice the first 5 columns for rate limitting & Run Model

Current model should be run with 5 columns x 25 rows

In [15]:
clean_data = clean_entire_dataframe(df2)

--- Cleaning Column: fdic_certificate_number ---

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Error in fdic_certificate_number at index 0: litellm.BadRequestError: GeminiException BadRequestError - {
  "error": {
    "code": 403,
    "message": "Your API key was reported as leaked. Please use another API key.",
    "status": "PERMISSION_DENIED"
  }
}

--- Cleaning Column: institution_name ---

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Error in institution_name at index 0: litellm.BadRequestError: GeminiException BadRequestError - {
  "error": {
    "code": 403,
    "message": "Your API key was reported as leaked. Please use another API key.",
    "status": "PERMISSION_DENIED"
  }
}



KeyboardInterrupt: 

In [61]:
#Expected (25, 5)
df_fully_cleaned.shape

(25, 5)

In [ ]:
clean_data.to_csv('../data/fdic/byllm_cleaned_fdic.csv', index=False)